In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

# **Parameters**

Parameter config for building the fact table.
- **catalog / source_schema / source_object**: where the source bookings (silver) live
- **cdc_column**: change-tracking column (`modifiedDate`) used to find new or changed rows
- **backdated_refresh**: optional date to reload from; leave empty for a normal incremental run
- **target_schema / target_object**: where the fact table (gold) is written
- **fact_key_cols**: columns that uniquely identify a fact row, used as the MERGE match condition (the dimension surrogate keys plus `booking_date`)
- **dimensions**: one entry per dimension to join. Each has the table name, an alias, and `join_keys` as `(fact_col, dim_col)` pairs. Add more pairs for composite keys.
- **fact_columns**: columns kept from the source besides the surrogate keys

In [0]:
# Catalog Name
catalog = "databricksete"

# Source Schema
source_schema = "silver"

# Source Object 
source_object = "silver_bookings"

# CDC Column
cdc_column = "modifiedDate"

# Backdated Refresh
backdated_refresh = ""

# Source Fact Table
fact_table = f"{catalog}.{source_schema}.{source_object}"

# Target Schema 
target_schema = "gold"

# Target Object 
target_object = "FactBookings"

# Fact Key Cols List 
fact_key_cols = ["DimPassengersKey","DimFlightsKey","DimAirportsKey","booking_date"]


In [0]:
dimensions = [
    {
        "table": f"{catalog}.{target_schema}.DimPassengers",
        "alias": "DimPassengers",
        "join_keys": [("passenger_id", "passenger_id")]  # (fact_col, dim_col)
    },
    {
        "table": f"{catalog}.{target_schema}.DimFlights",
        "alias": "DimFlights",
        "join_keys": [("flight_id", "flight_id")]  # (fact_col, dim_col)
    },
    {
        "table": f"{catalog}.{target_schema}.DimAirports",
        "alias": "DimAirports",
        "join_keys": [("airport_id", "airport_id")]  # (fact_col, dim_col)
    },
]


# Columns to keep from Fact table apart from surrogate keys
fact_columns = ["amount","booking_date","modifiedDate"]

# **Last Load Date**

Works out where the incremental load should start from.
- Normal run: `last_load` is the max `cdc_column` already in the fact table.
- If the fact table doesn't exist yet, `last_load` is set to `1900-01-01`, so the first run loads everything.
- If `backdated_refresh` is given, it overrides `last_load` to reload from a set date.

In [0]:
# No Back Dated Refresh
if len(backdated_refresh) == 0:
  
  # If the target table exists
  if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    last_load = spark.sql(f"SELECT max({cdc_column}) FROM {catalog}.{target_schema}.{target_object}").collect()[0][0]

  # If trhe table doesn't exist, then perform full load  
  else:
    last_load = "1900-01-01 00:00:00"

# Yes Back Dated Refresh
else:
  last_load = backdated_refresh

# **DYNAMIC FACT QUERY [BRING KEYS]**

Builds the SQL that swaps business ids for dimension surrogate keys and returns a dynamic SQL string.
- Starts with the fact columns from the source (`f.amount`, `f.booking_date`, ...).
- Loops over `dimensions`. For each one it adds the dimension's surrogate key to the SELECT and builds a `LEFT JOIN` on the business keys (e.g. `f.passenger_id = DimPassengers.passenger_id`).
- `LEFT JOIN` keeps bookings even when no dimension row matches (that surrogate key is NULL).
- Filters to rows where `cdc_column >= last_load`, so only new or changed bookings are processed.
- `df_fact = spark.sql(query)` runs the generated dynamic sql query.

In [0]:
def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, processing_date):
    fact_alias = "f"
    
    # Base columns to select
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    # Build joins dynamically
    join_clauses = []

    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]
        table_name = table_full.split('.')[-1]
        surrogate_key = f"{alias}.{table_name}Key"
        select_cols.append(surrogate_key)

        # Build ON clause
        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}" for fk, dk in dim["join_keys"]
        ]

        join_clause = f"LEFT JOIN {table_full} {alias} ON " + " AND ".join(on_conditions)
        join_clauses.append(join_clause)

    # Final SELECT and JOIN clauses
    select_clause = ",\n".join(select_cols)
    joins = "\n".join(join_clauses)

    # WHERE clause for incremental filtering
    where_clause = f"{fact_alias}.{cdc_column} >= DATE('{last_load}')"

    # Final query
    query = f"""
            SELECT {select_clause}
            FROM {fact_table} {fact_alias}
            {joins}
            WHERE {where_clause}
            """.strip()

    return query


In [0]:
query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, last_load)
df_fact = spark.sql(query)

# **UPSERT**

Writes `df_fact` to the fact table.
- Builds the match condition from `fact_key_cols` (e.g. `src.DimPassengersKey = trg.DimPassengersKey AND ...`).
- **Fact table exists**: runs a Delta `MERGE`. Matching rows are updated only if the incoming `cdc_column` is the same or newer than the existing one, so older data can't overwrite newer data. Non-matching rows are inserted.
- **Fact table doesn't exist (first run)**: the table is created by writing `df_fact` in append mode with `saveAsTable`.

In [0]:
# Fact Key Columns Merge Condition
fact_key_cols_str = " AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])
fact_key_cols_str

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_fact.alias("src"), fact_key_cols_str)\
                        .whenMatchedUpdateAll(condition = f"src.{cdc_column} >= trg.{cdc_column}")\
                        .whenNotMatchedInsertAll()\
                        .execute()

else: 

    df_fact.write.format("delta")\
            .mode("append")\
            .saveAsTable(f"{catalog}.{target_schema}.{target_object}")
